# 5 — Integration

Runnable code from chapter 5 of *AI Agents*, generated from the book's own sources.

Cells follow the order of the chapter, and the headings below carry the book's section and listing numbers, so you can read and run side by side.

Some cells set up state the book does not print — imports, the API client, helpers introduced earlier. They are included so the notebook runs on its own, and are marked *setup*. Run it from top to bottom.

Your results will differ in wording from the printed ones: these are live model calls.

## 5.3 Serving other software

*setup — not printed in the book*

In [ ]:
import os
import json
import warnings
from dotenv import load_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore")
load_dotenv()

client = OpenAI()
CHAT_MODEL = os.environ["CHAT_MODEL"]

**Listing 5.1** — The triage capability exposed as an API service with a typed contract

In [ ]:
from typing import Literal

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class TriageRequest(BaseModel):
    subject: str
    body: str

class TriageResult(BaseModel):
    category: Literal[
        "billing", "technical", "account", "other"]
    priority: Literal["low", "normal", "urgent"]
    draft_reply: str

@app.post("/triage", response_model=TriageResult)
def triage(req: TriageRequest) -> TriageResult:
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": (
                "You triage customer support emails. "
                "Return valid, minified JSON with keys "
                "category (billing, technical, account, "
                "other), priority (low, normal, urgent), "
                "and draft_reply (one short, polite "
                "paragraph).")},
            {"role": "user", "content": (
                f"Subject: {req.subject}\n\n{req.body}")},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
        seed=42,
        max_tokens=300,
    )
    return TriageResult(
        **json.loads(resp.choices[0].message.content)
    )

In [ ]:
from fastapi.testclient import TestClient

api = TestClient(app)
r = api.post("/triage", json={
    "subject": "Charged twice for March",
    "body": ("Hi, my card was charged twice for the "
             "March invoice. Please fix this."),
})
print(r.status_code)
print(json.dumps(r.json(), indent=2))

## 5.4 Acting on events

**Listing 5.2** — An idempotent webhook endpoint for `ticket-created` events

In [ ]:
processed_events = set()

@app.post("/webhooks/ticket-created")
def on_ticket_created(event: dict):
    event_id = event["event_id"]
    if event_id in processed_events:
        return {"status": "duplicate, ignored"}
    processed_events.add(event_id)  # before, not after

    result = triage(TriageRequest(**event["ticket"]))
    return {"status": "processed",
            "category": result.category,
            "priority": result.priority}

In [ ]:
event = {
    "event_id": "evt-20260703-001",
    "ticket": {
        "subject": "Cannot log in since this morning",
        "body": "Password reset emails never arrive.",
    },
}
print(api.post("/webhooks/ticket-created", json=event).json())
print(api.post("/webhooks/ticket-created", json=event).json())